# Semana 4: Práctica. El DCF de SCCO, versión profesional

**Curso:** Tópicos de Finanzas Avanzadas (ECON-421, UPAO 2026-20)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JonathanRosasV/topicos-finanzas-upao/blob/main/04_valor_terminal_ev/clase04_practica.ipynb)

Retomamos el DCF de Southern Copper de la semana pasada y lo llevamos a estándar profesional: valor terminal por dos métodos, puente completo al equity y análisis de sensibilidad con mapa de calor.

**Requisito previo:** `git pull` en tu fork para tener el `utils/finanzas.py` actualizado.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

from utils.finanzas import (capm, wacc, fcff_desde_ebit, valor_terminal,
                            valor_terminal_multiplo, dcf_dos_etapas)

TICKER = "SCCO"
tk = yf.Ticker(TICKER)

def fila(df, *nombres):
    for n in nombres:
        if n in df.index:
            return df.loc[n]
    raise KeyError(nombres)

## 1. Reconstruir la base de la semana 3

Recuperamos en pocas celdas el FCFF, el WACC y la proyección que armamos la semana pasada (mismo código, condensado).

In [ ]:
est, bal, cf = tk.income_stmt, tk.balance_sheet, tk.cashflow

ebit = fila(est, "EBIT", "Operating Income")
interes = fila(est, "Interest Expense").abs()
t_ef = float((fila(est, "Tax Provision") / fila(est, "Pretax Income")).iloc[0])
dep = fila(cf, "Depreciation And Amortization", "Depreciation Amortization Depletion", "Depreciation")
capex = fila(cf, "Capital Expenditure").abs()
wcinv = -fila(cf, "Change In Working Capital")

fcff_0 = fcff_desde_ebit(float(ebit.iloc[0]), t_ef, float(dep.iloc[0]),
                         float(capex.iloc[0]), float(wcinv.iloc[0])) / 1e6

# WACC con los supuestos declarados de la semana 2
import statsmodels.api as sm
px = yf.download([TICKER, "^GSPC"], start="2021-08-01",
                 interval="1mo", auto_adjust=True, progress=False)["Close"]
r = px.pct_change().dropna().rename(columns={"^GSPC": "SP500"})
modelo = sm.OLS(r[TICKER], sm.add_constant(r["SP500"]), missing="drop").fit()
beta_aj = 0.67 * float(modelo.params["SP500"]) + 0.33

rf, ERP, t_marginal = 0.045, 0.055, 0.30   # Peru 29.5% y Mexico 30%, igual que en la semana 3
ke = capm(rf, beta_aj, ERP)
E = tk.fast_info["marketCap"] / 1e6
D = float(fila(bal, "Total Debt").iloc[0]) / 1e6
kd = float(interes.iloc[0]) / 1e6 / D
WACC = wacc(E, D, ke, kd, t_marginal)

# Proyeccion declarada (la de la semana 3)
g_proy, g_perp = [0.06, 0.055, 0.05, 0.045, 0.04], 0.03
proy, f = [], fcff_0
for g in g_proy:
    f *= (1 + g)
    proy.append(f)

print(f"FCFF_0 = {fcff_0:,.0f} MM | WACC = {WACC:.2%} | FCFF_5 = {proy[-1]:,.0f} MM")

## 2. Valor terminal por los dos métodos

Método 1: perpetuidad con el g perpetuo declarado. Método 2: múltiplo de salida EV/EBITDA. Para el múltiplo usamos un valor declarado como supuesto (la semana 5 aprenderemos a justificarlo con comparables).

In [ ]:
ebitda = fila(est, "EBITDA", "Normalized EBITDA")
ebitda_0 = float(ebitda.iloc[0]) / 1e6
ebitda_5 = ebitda_0
for g in g_proy:
    ebitda_5 *= (1 + g)

MULTIPLO_SALIDA = 8.0    # supuesto declarado; justificar con comparables en la semana 5

VT_gordon = valor_terminal(proy[-1], WACC, g_perp)
VT_multiplo = valor_terminal_multiplo(ebitda_5, MULTIPLO_SALIDA)

print(f"EBITDA del anio 5            = {ebitda_5:,.0f} MM")
print(f"VT por perpetuidad (Gordon)  = {VT_gordon:,.0f} MM")
print(f"VT por multiplo de salida    = {VT_multiplo:,.0f} MM")
print(f"Multiplo implicito del Gordon = {VT_gordon / ebitda_5:.1f}x  (contra {MULTIPLO_SALIDA:.0f}x declarado)")

In [ ]:
# Referencia de mercado: a que multiplo cotiza la empresa hoy
caja = float(fila(bal, "Cash And Cash Equivalents",
                  "Cash Cash Equivalents And Short Term Investments").iloc[0]) / 1e6
EV_mercado = E + D - caja
print(f"EV de mercado = {EV_mercado:,.0f} MM | EV/EBITDA actual = {EV_mercado / ebitda_0:.1f}x")

**Pregunta de discusión:** ¿el múltiplo implícito de tu Gordon está dentro del rango al que cotizan las mineras grandes? Si difiere mucho del múltiplo declarado, ¿qué supuesto revisarías primero?

Mira también el múltiplo al que cotiza la empresa hoy (celda anterior). Si es mucho mayor que el de salida, no corras a subir el múltiplo declarado: el de salida corresponde a la empresa ya madura del año 5 y suele ser menor que el actual. Usar el múltiplo actual de una empresa que el mercado todavía ve en crecimiento es el error común 5 de las láminas.

## 3. EV y puente al equity por ambos métodos

In [ ]:
acciones = tk.fast_info["shares"] / 1e6
px_mercado = tk.fast_info["lastPrice"]

resumen = {}
for nombre, vt in [("Gordon", VT_gordon), (f"Multiplo {MULTIPLO_SALIDA:.0f}x", VT_multiplo)]:
    res = dcf_dos_etapas(proy, WACC, vt)
    eq = res["valor"] - (D - caja)
    resumen[nombre] = {"EV (MM)": res["valor"], "peso VT": res["peso_vt"],
                       "equity (MM)": eq, "por accion": eq / acciones,
                       "vs precio": eq / acciones / px_mercado - 1}

pd.DataFrame(resumen).T.round(2)

## 4. La tabla de sensibilidad

El corazón de la clase: EV para una grilla de WACC y g perpetuo. Construimos la tabla con un doble bucle y la mostramos como mapa de calor.

In [ ]:
waccs = np.round(np.arange(WACC - 0.01, WACC + 0.0125, 0.005), 4)
gs = [0.02, 0.025, 0.03, 0.035, 0.04]

tabla = pd.DataFrame(index=waccs, columns=gs, dtype=float)
for w in waccs:
    for gg in gs:
        vt = valor_terminal(proy[-1], w, gg)
        eq = dcf_dos_etapas(proy, w, vt)["valor"] - (D - caja)
        tabla.loc[w, gg] = eq / acciones

tabla.index = [f"{w:.2%}" for w in waccs]
tabla.columns = [f"{g:.1%}" for g in gs]
tabla.index.name, tabla.columns.name = "WACC", "g perpetuo"
tabla.round(2)

In [ ]:
# Color segun la diferencia contra el precio de mercado:
# verde = el modelo vale mas que el precio (subvaluada), rojo = vale menos (sobrevaluada)
dif = tabla.values.astype(float) / px_mercado - 1
lim = np.abs(dif).max()

fig, ax = plt.subplots(figsize=(8, 4.5))
im = ax.imshow(dif, cmap="RdYlGn", vmin=-lim, vmax=lim, aspect="auto")
ax.set_xticks(range(len(tabla.columns)), tabla.columns)
ax.set_yticks(range(len(tabla.index)), tabla.index)
ax.set_xlabel("g perpetuo"); ax.set_ylabel("WACC")
for i in range(tabla.shape[0]):
    for j in range(tabla.shape[1]):
        ax.text(j, i, f"{tabla.values[i, j]:.1f}", ha="center", va="center", fontsize=9)
ax.set_title(f"{TICKER}: valor por accion segun WACC y g (precio de mercado: {px_mercado:.1f})")
plt.tight_layout(); plt.show()

subvaluada = int((tabla.values.astype(float) > px_mercado).sum())
print(f"Celdas donde el modelo supera al precio: {subvaluada} de {tabla.size}")
print(f"Rango del modelo: {tabla.values.min():,.2f} a {tabla.values.max():,.2f} | precio: {px_mercado:,.2f}")

**Lectura del mapa:** cada celda es una valoración defendible con supuestos razonables. ¿En cuántas celdas la acción está subvaluada y en cuántas sobrevaluada? Esa es la respuesta honesta de tu modelo: un rango, no un veredicto de una sola celda.

**Cuando el precio cae fuera del mapa.** Si ninguna celda alcanza el precio de mercado (o todas lo superan), el mapa dice algo más fuerte: la discrepancia con el mercado no está en el WACC ni en el g dentro de lo defendible, sino en los flujos: el flujo base, el crecimiento de la etapa explícita o, en una minera, el precio del cobre que el mercado está proyectando. Es la conclusión del DCF inverso de la semana 3 vista desde otro ángulo, y es un hallazgo legítimo para un informe: dice qué habría que creer para justificar el precio.

## 5. Cierre

Desde hoy quedan en la librería del curso: `valor_terminal()`, `valor_terminal_multiplo()` y `dcf_dos_etapas()`. Con esto la caja de herramientas DCF está completa.

**Tarea de la semana** (`clase04_tarea.ipynb`): replicar todo esto con tu empresa. Entrega hasta el lunes de la semana siguiente, vía commit en tu fork.

**TR1 (07/10):** informe de valoración completo; la consigna y la rúbrica quedan publicadas en el repositorio esta semana.

**Próxima semana:** valoración por múltiplos y comparables.